# B21 BSA dilution series with server-side HDF5 chunks

This notebook initializes the Diamond Light Source B21 example. It inventories the donated batch, validates the relative Eiger links without loading full detector images, prepares a 21-frame chunk schedule, and starts a MoDaCor runtime.

The processing pipeline is deliberately not defined yet. Once the correction strategy is specified, the execution section will register each changing measurement as an HDF source and bind the detector and aligned metadata to each chunk selection. This keeps the large detector reads inside the runtime; the notebook will not upload image arrays through a buffer.


## Configuration

Run from the examples repository root or from this directory. The default five-frame chunks split every 21-frame acquisition into four full chunks and one one-frame edge chunk.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Start Jupyter from MoDaCor_examples or one of its subdirectories."
    )

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("DLS/B21")
DATA_DIR = PROJECT_DIR / "data"
WORK_DIR = PROJECT_DIR / "work" / "chunk_server"


In [ ]:
import atexit

import h5py
import hdf5plugin
import modacor
from IPython.display import JSON, display
from modacor.client import LocalRuntimeServer

DETECTOR_PATH = "/entry1/instrument/detector/data"
CHUNK_SIZE = 5
SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
EXPECTED_TITLES = (
    "buffer_1",
    "bsa_10mgml",
    "bsa_5mgml",
    "bsa_2p5mgml",
    "bsa_1p25mgml",
    "bsa_0p6mgml",
    "bsa_0p3mgml",
    "buffer_2",
)


## Discover and validate the raw batch

Only metadata, link targets, shapes, and dtypes are inspected here. Accessing the detector dataset resolves the relative NeXus-to-Eiger link but does not materialize its image stack.


In [ ]:
def text_value(dataset):
    value = dataset.asstr()[()]
    return str(value.item() if getattr(value, "shape", None) == () else value)


runs = []
for master_path in sorted(DATA_DIR.glob("b21-*.nxs")):
    with h5py.File(master_path, "r") as nexus:
        title = text_value(nexus["/entry1/title"])
        detector = nexus[DETECTOR_PATH]
        detector_group = nexus["/entry1/instrument/detector"]
        link = detector_group.get("data", getlink=True)
        if not isinstance(link, h5py.ExternalLink):
            raise TypeError(f"{master_path.name}: detector data is not an external link")
        target = master_path.parent / link.filename
        if not target.is_file():
            raise FileNotFoundError(f"{master_path.name}: missing {link.filename}")
        runs.append(
            {
                "run": master_path.stem,
                "title": title,
                "role": "buffer" if title.startswith("buffer") else "sample",
                "master_path": master_path,
                "detector_shape": tuple(detector.shape),
                "detector_dtype": str(detector.dtype),
                "eiger_target": target.name,
            }
        )

titles = tuple(run["title"] for run in runs)
if titles != EXPECTED_TITLES:
    raise ValueError(f"Unexpected B21 run order or titles: {titles}")
if {run["detector_shape"][:2] for run in runs} != {(1, 21)}:
    raise ValueError("Expected one 21-frame acquisition in every raw master.")

display(
    JSON(
        [
            {key: value for key, value in run.items() if key != "master_path"}
            for run in runs
        ]
    )
)


## Prepare direct-HDF source registrations and chunk schedule

The source descriptions below are ready for `SessionClient.register_source`. The future chunk plan must bind `sample` plus `DETECTOR_PATH` as an aligned source and use selections of the form `(all, start:stop, all, all)`. Static calibration or mask inputs should receive explicit static bindings once their exact paths are chosen.


In [ ]:
def hdf_source_registration(master_path):
    return {
        "ref": "sample",
        "type": "hdf",
        "location": str(Path(master_path).resolve()),
    }


work_items = []
for measurement_index, run in enumerate(runs):
    frame_count = run["detector_shape"][1]
    for chunk_index, start in enumerate(range(0, frame_count, CHUNK_SIZE)):
        stop = min(start + CHUNK_SIZE, frame_count)
        work_items.append(
            {
                "chunk_id": f"m{measurement_index:03d}-c{chunk_index:03d}",
                "run": run["run"],
                "title": run["title"],
                "start": start,
                "stop": stop,
                "input_shape": (1, stop - start, *run["detector_shape"][2:]),
                "source": hdf_source_registration(run["master_path"]),
            }
        )

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Measurements: {len(runs)}")
print(f"Chunks: {len(work_items)} ({len(work_items) // len(runs)} per measurement)")
display(JSON(work_items[:5]))


## Start or reuse the MoDaCor runtime

This establishes the server boundary now. Session construction and chunk execution remain intentionally absent until the B21 processing pipeline and output schema are defined.


In [ ]:
WORK_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=WORK_DIR / "modacor_server.log",
    environment={"HDF5_PLUGIN_PATH": hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")


## Processing handoff

The next iteration will add the MoDaCor pipeline and complete chunk plan here. Decisions still needed are: how the two bracketing buffers are used; which incident-flux, transmission, exposure-time, calibration, and mask datasets are authoritative; whether frame rejection or averaging occurs before or after corrections; and which DAWN outputs form the numerical comparison.

After those decisions, this section will create one runtime session, register the static sources, patch the `sample` HDF source as work advances from run to run, submit typed chunk specifications, and finalize one chunked HDF5 output below `work/chunk_server/`.


## Cleanup


In [ ]:
server.stop()
print(
    "Stopped the notebook-owned runtime."
    if not client.is_ready()
    else "Left the external runtime running."
)
